In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    roc_curve,
    precision_recall_curve,
    ConfusionMatrixDisplay
)

: 

In [ ]:
# ---------------------------------------------------------
# 1. Load data
# ---------------------------------------------------------
# Option A: local file
# df = pd.read_csv("credit_risk_dataset.csv")

# Option B: from GitHub/raw URL
url = "https://raw.githubusercontent.com/DaneshBU/CS506-Project/main/data/credit_risk_dataset.csv"
df = pd.read_csv(url)

print("Original dataset shape:", df.shape)
print(df.head())

In [ ]:
# ---------------------------------------------------------
# 2. Basic cleaning
# ---------------------------------------------------------
df = df.drop_duplicates().copy()
print("\nAfter removing duplicates:", df.shape)

# target column
target_col = "loan_status"

# quick safety check
if target_col not in df.columns:
    raise ValueError(f"Target column '{target_col}' not found in dataset.")

# Separate features and target
X = df.drop(columns=[target_col])
y = df[target_col]

print("\nTarget distribution:")
print(y.value_counts(normalize=True))

In [ ]:
# ---------------------------------------------------------
# 3. Detect numeric and categorical columns
# ---------------------------------------------------------
numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object"]).columns.tolist()

print("\nNumeric features:", numeric_features)
print("Categorical features:", categorical_features)

In [ ]:
# ---------------------------------------------------------
# 4. Preprocessing pipelines
# ---------------------------------------------------------
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

In [ ]:
# ---------------------------------------------------------
# 5. Split into train / validation / test
# ---------------------------------------------------------
# 60% train, 20% validation, 20% test
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val,
    test_size=0.25,   # 25% of 80% = 20% overall
    random_state=42,
    stratify=y_train_val
)

print("\nTrain shape:", X_train.shape)
print("Validation shape:", X_val.shape)
print("Test shape:", X_test.shape)

In [ ]:
# ---------------------------------------------------------
# 6. Build XGBoost model
# ---------------------------------------------------------
model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", XGBClassifier(
        n_estimators=300,
        max_depth=5,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=42
    ))
])

In [ ]:
# ---------------------------------------------------
# 7. Train model
# ---------------------------------------------------
model.fit(X_train, y_train)


In [ ]:
# ---------------------------------------------------
# 8. Predict probabilities on validation set
# ---------------------------------------------------
val_probs = model.predict_proba(X_val)[:, 1]

In [ ]:
# ---------------------------------------------------------
# 9. Threshold tuning on validation set
# ---------------------------------------------------------
thresholds = np.arange(0.10, 0.91, 0.05)

threshold_results = []

for t in thresholds:
    val_preds_t = (val_probs >= t).astype(int)
    threshold_results.append({
        "threshold": t,
        "accuracy": accuracy_score(y_val, val_preds_t),
        "precision": precision_score(y_val, val_preds_t, zero_division=0),
        "recall": recall_score(y_val, val_preds_t, zero_division=0),
        "f1": f1_score(y_val, val_preds_t, zero_division=0)
    })

threshold_df = pd.DataFrame(threshold_results)
print("\nThreshold tuning results:")
print(threshold_df)

# Choose threshold with best F1 score
best_row = threshold_df.loc[threshold_df["f1"].idxmax()]
best_threshold = best_row["threshold"]

print("\nBest threshold based on validation F1:", best_threshold)
print(best_row)

In [ ]:
# ---------------------------------------------------------
# 10. Validation evaluation at best threshold
# ---------------------------------------------------------
val_preds = (val_probs >= best_threshold).astype(int)

print("\n=== Validation Metrics ===")
print(f"Threshold : {best_threshold:.2f}")
print("Accuracy  :", accuracy_score(y_val, val_preds))
print("Precision :", precision_score(y_val, val_preds, zero_division=0))
print("Recall    :", recall_score(y_val, val_preds, zero_division=0))
print("F1 Score  :", f1_score(y_val, val_preds, zero_division=0))
print("ROC AUC   :", roc_auc_score(y_val, val_probs))

print("\nValidation Classification Report:")
print(classification_report(y_val, val_preds, zero_division=0))

In [ ]:
# ---------------------------------------------------------
# 11. ROC Curve (Validation)
# ---------------------------------------------------------
fpr, tpr, roc_thresholds = roc_curve(y_val, val_probs)
val_auc = roc_auc_score(y_val, val_probs)

plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr, label=f"Logistic Regression (AUC = {val_auc:.3f})")
plt.plot([0, 1], [0, 1], linestyle="--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve - Validation Set")
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
# ---------------------------------------------------------
# 12. Precision-Recall Curve (Validation)
# ---------------------------------------------------------
precisions, recalls, pr_thresholds = precision_recall_curve(y_val, val_probs)

plt.figure(figsize=(7, 5))
plt.plot(recalls, precisions)
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve - Validation Set")
plt.grid(True)
plt.show()


In [ ]:
# ---------------------------------------------------------
# 13. Confusion Matrix (Validation)
# ---------------------------------------------------------
cm_val = confusion_matrix(y_val, val_preds)
disp = ConfusionMatrixDisplay(confusion_matrix=cm_val)
disp.plot()
plt.title(f"Validation Confusion Matrix (Threshold = {best_threshold:.2f})")
plt.show()

In [ ]:
# ---------------------------------------------------------
# 14. Final test evaluation using chosen threshold
# ---------------------------------------------------------
test_probs = model.predict_proba(X_test)[:, 1]
test_preds = (test_probs >= best_threshold).astype(int)

print("\n=== Test Metrics ===")
print(f"Threshold : {best_threshold:.2f}")
print("Accuracy  :", accuracy_score(y_test, test_preds))
print("Precision :", precision_score(y_test, test_preds, zero_division=0))
print("Recall    :", recall_score(y_test, test_preds, zero_division=0))
print("F1 Score  :", f1_score(y_test, test_preds, zero_division=0))
print("ROC AUC   :", roc_auc_score(y_test, test_probs))

print("\nTest Classification Report:")
print(classification_report(y_test, test_preds, zero_division=0))

In [ ]:
# ---------------------------------------------------------
# 15. Confusion Matrix (Test)
# ---------------------------------------------------------
cm_test = confusion_matrix(y_test, test_preds)
disp = ConfusionMatrixDisplay(confusion_matrix=cm_test)
disp.plot()
plt.title(f"Test Confusion Matrix (Threshold = {best_threshold:.2f})")
plt.show()

In [ ]:
# ---------------------------------------------------------
# 16. Sample prediction output
# ---------------------------------------------------------
test_results = X_test.copy()
test_results["actual_default"] = y_test.values
test_results["predicted_probability"] = test_probs
test_results["predicted_class"] = test_preds

print("\nSample prediction results:")
print(test_results[["actual_default", "predicted_probability", "predicted_class"]].head(10))

# Save sample predictions if needed
test_results.to_csv("logistic_regression_test_predictions.csv", index=False)

In [ ]:
# ---------------------------------------------------------
# 17. Feature importance from XGBoost
# ---------------------------------------------------------
ohe = model.named_steps["preprocessor"].named_transformers_["cat"].named_steps["onehot"]
encoded_cat_names = ohe.get_feature_names_out(categorical_features)

all_feature_names = numeric_features + list(encoded_cat_names)
importances = model.named_steps["classifier"].feature_importances_

importance_df = pd.DataFrame({
    "feature": all_feature_names,
    "importance": importances
}).sort_values("importance", ascending=False)

print("\nTop 20 most influential features:")
print(importance_df.head(20))

importance_df.to_csv("xgboost_feature_importance.csv", index=False)

In [ ]:
# ---------------------------------------------------------
# 18. Plot top XGBoost feature importances
# ---------------------------------------------------------
top_features = importance_df.head(20).sort_values("importance", ascending=True)

plt.figure(figsize=(10, 6))
plt.barh(top_features["feature"], top_features["importance"])
plt.xlabel("Importance")
plt.title("Top XGBoost Feature Importances")
plt.grid(True, axis="x")
plt.show()

In [ ]:
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
param_grid = { "classifier__n_estimators": [200, 300, 500], "classifier__max_depth": [3, 4, 5, 6], "classifier__learning_rate": [0.03, 0.05, 0.1], "classifier__min_child_weight": [1, 3, 5], "classifier__subsample": [0.7, 0.8, 0.9, 1.0], "classifier__colsample_bytree": [0.7, 0.8, 0.9, 1.0], "classifier__gamma": [0, 0.1, 0.3], "classifier__reg_alpha": [0, 0.1, 1], "classifier__reg_lambda": [1, 2, 5] }
search = RandomizedSearchCV(
    estimator=model,
    param_distributions=param_grid,
    n_iter=25,
    scoring="f1",
    cv=cv,
    verbose=2,
    n_jobs=-1,
    random_state=42
)

search.fit(X_train, y_train)

print("Best parameters:")
print(search.best_params_)

print("\nBest CV score:")
print(search.best_score_)